In [21]:
import os
import pandas as pd


os.environ['KAGGLE_API_TOKEN'] = "KGAT_4f6c6a05cc84e354f031c5cfd079ac02"

!pip install -q kaggle

print("Setup complete! Your VS Code notebook is now connected to Kaggle.")

Setup complete! Your VS Code notebook is now connected to Kaggle.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:


!python -m kaggle datasets download -d ulrikthygepedersen/online-retail-dataset --unzip


import os
print("Downloaded files:", os.listdir('.'))

Dataset URL: https://www.kaggle.com/datasets/ulrikthygepedersen/online-retail-dataset
License(s): Attribution 4.0 International (CC BY 4.0)

Downloaded files: ['cleaned_ecommerce_data.csv', 'data_cleaning.ipynb', 'online_retail.csv']



  0%|          | 0.00/7.38M [00:00<?, ?B/s]
 14%|█▎        | 1.00M/7.38M [00:01<00:07, 876kB/s]
 27%|██▋       | 2.00M/7.38M [00:01<00:03, 1.73MB/s]
 41%|████      | 3.00M/7.38M [00:01<00:01, 2.73MB/s]
 68%|██████▊   | 5.00M/7.38M [00:01<00:00, 5.06MB/s]
100%|██████████| 7.38M/7.38M [00:01<00:00, 7.84MB/s]
100%|██████████| 7.38M/7.38M [00:01<00:00, 4.29MB/s]


In [23]:
import pandas as pd



df = pd.read_csv('online_retail.csv')


print("--- Missing Values Per Column ---")
print(df.isnull().sum())

--- Missing Values Per Column ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [24]:
df['TotalSales'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'TotalSales']].head()

,Quantity,UnitPrice,TotalSales
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [25]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df[['InvoiceDate','Year','Month','DayOfWeek']].head()

,InvoiceDate,Year,Month,DayOfWeek
0,2010-12-01 08:26:00,2010,12,2
1,2010-12-01 08:26:00,2010,12,2
2,2010-12-01 08:26:00,2010,12,2
3,2010-12-01 08:26:00,2010,12,2
4,2010-12-01 08:26:00,2010,12,2


In [26]:
df.to_csv('cleaned_ecommerce_data.csv',index = False)
print("Success! 'cleaned_ecommerce_data.csv' has been created and saved to your folder.")

Success! 'cleaned_ecommerce_data.csv' has been created and saved to your folder.


In [27]:
!pip install -q pymysql cryptography


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# !pip install sqlalchemy

In [30]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql
# 1. Enter your local MySQL Connection Details here:
DB_USER = "root"          # Your MySQL username (usually root)
DB_PASSWORD = "Mendonsa31!"  # Your MySQL password
DB_HOST = "localhost"      # Usually localhost
DB_PORT = "3307"          # Default MySQL port is 3306
DB_NAME = "ecommerce_db"   # The name of the database you want to create/use

# 2. Create a connection engine using SQLAlchemy (required by pandas for MySQL)
# We use pymysql as the underlying driver
conn = pymysql.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD, port=int(DB_PORT))
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME};")
conn.close()
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# 3. Read your cleaned CSV data
df_cleaned = pd.read_csv('cleaned_ecommerce_data.csv')

# 4. Push the data straight into your MySQL instance as a table named 'sales'
# chunksize speeds up the transfer for larger datasets
df_cleaned.to_sql(name='sales', con=engine, if_exists='replace', index=False, chunksize=5000)

print(f"Success! Your data is now living inside the '{DB_NAME}' database in the 'sales' table.")

Success! Your data is now living inside the 'ecommerce_db' database in the 'sales' table.


In [32]:
# Write the MySQL query as a Python string
mysql_query = """
WITH MonthlySales AS (
    SELECT 
        DATE_FORMAT(InvoiceDate, '%%Y-%%m') AS sales_month,
        SUM(TotalSales) AS total_revenue
    FROM sales
    GROUP BY 1
)
SELECT sales_month, total_revenue FROM MonthlySales ORDER BY sales_month;
"""

# Execute the query using your MySQL engine connection
monthly_trends_df = pd.read_sql_query(mysql_query, con=engine)
monthly_trends_df

,sales_month,total_revenue
0,2010-12,748957.020
1,2011-01,560000.260
2,2011-02,498062.650
3,2011-03,683267.080
4,2011-04,493207.121
5,2011-05,723333.510
6,2011-06,691123.120
7,2011-07,681300.111
8,2011-08,682680.510
9,2011-09,1019687.622
